# 04. 管道与重定向

AGDR VLA 入门教程 · 第 2 章 · 第 1 周

- **学习目标**：理解三种标准流，能把输出存入文件，能用管道把小命令组合起来
- **前置知识**：[03-查看与搜索](./03-查看与搜索.ipynb)
- **预计学习时长**：1 小时

## 三种标准流

每个命令运行时都有三条数据通道：

| 名称 | 编号 | 默认去向 | 用途 |
| --- | --- | --- | --- |
| 标准输入 stdin | 0 | 来自键盘 | 命令读取的内容 |
| 标准输出 stdout | 1 | 屏幕 | 正常结果 |
| 标准错误 stderr | 2 | 屏幕 | 错误信息 |

stdout 和 stderr 默认都显示在屏幕上，看起来一样，但其实是两条通道，可以分开处理。

## 重定向：把输出存入文件

| 写法 | 作用 |
| --- | --- |
| `命令 > 文件` | stdout 写入文件，**覆盖**原内容 |
| `命令 >> 文件` | stdout **追加**到文件末尾 |
| `命令 2> 文件` | stderr 写入文件 |
| `命令 > 文件 2>&1` | stdout 和 stderr 都写入同一个文件 |
| `命令 > /dev/null` | 丢弃输出，`/dev/null` 是系统的「黑洞」 |
| `命令 < 文件` | 从文件读取 stdin |

In [ ]:
%%bash
rm -rf /tmp/linux-demo && mkdir -p /tmp/linux-demo && cd /tmp/linux-demo
echo "第一行" > out.txt
echo "第二行" >> out.txt
cat out.txt
echo "--- 再用一次 >"
echo "第三行" > out.txt
cat out.txt

> ⚠️ `>` 会先清空文件，再写入。写错文件名可能直接清空一个重要文件，不确定时用 `>>`。

下面的命令同时产生正常输出和错误，我们把两者分开：

In [ ]:
%%bash
cd /tmp/linux-demo
touch exists.txt
ls exists.txt missing.txt > ok.txt 2> err.txt
echo "--- ok.txt"
cat ok.txt
echo "--- err.txt"
cat err.txt

训练时最常见的写法是把所有输出存到日志文件：

```bash
python train.py > train.log 2>&1
```

`2>&1` 的意思是「把 stderr 送到 stdout 现在的去向」。顺序不能反：`2>&1 > train.log` 的效果不一样，错误信息仍然会显示在屏幕上。

## 管道：把命令串起来

`|` 把前一个命令的 stdout 接到后一个命令的 stdin。Linux 的思路是**让每个小工具只做好一件事，再用管道组合**。

```text
grep INFO train.log | wc -l
   找出 INFO 行     →  数一数有几行
```

> 管道只传递 stdout，stderr 仍然显示在屏幕上。需要一起传时写 `2>&1 |`。

## 管道里的常用工具

| 命令 | 作用 |
| --- | --- |
| `sort` | 排序（默认按文本）；`-n` 按数字，`-r` 倒序 |
| `uniq` | 合并**相邻**的重复行；`-c` 同时计数 |
| `cut -d= -f2` | 以 `=` 分隔，取第 2 列 |
| `tr -s ' '` | 把连续的空格压缩成一个 |
| `head` / `tail` | 取前几行 / 后几行 |
| `wc -l` | 数行 |
| `tee 文件` | 既显示在屏幕上，又存入文件 |

逐步搭一个管道：从日志里找出最低的 loss。**先写一段，看一下输出，再接下一段**，这是写管道的正确方式。

In [ ]:
%%bash
cd /tmp/linux-demo
cat > train.log <<'EOF'
[INFO] epoch=1 loss=2.31
[WARN] gpu_memory_high
[INFO] epoch=2 loss=10.5
[ERROR] dataloader_timeout
[INFO] epoch=3 loss=1.42
[ERROR] camera_frame_dropped
[INFO] epoch=4 loss=0.98
[ERROR] dataloader_timeout
EOF
grep -o 'loss=[0-9.]*' train.log

In [ ]:
%%bash
cd /tmp/linux-demo
grep -o 'loss=[0-9.]*' train.log | cut -d= -f2

In [ ]:
%%bash
cd /tmp/linux-demo
echo "--- 按文本排序（10.5 排错了位置）"
grep -o 'loss=[0-9.]*' train.log | cut -d= -f2 | sort
echo "--- 按数字排序"
grep -o 'loss=[0-9.]*' train.log | cut -d= -f2 | sort -n
echo "--- 最低的 loss"
grep -o 'loss=[0-9.]*' train.log | cut -d= -f2 | sort -n | head -n 1

`sort` 默认按文本比较，`10.5` 会排在 `2.31` 前面；比较数字要加 `-n`。

### 统计「谁出现得最多」

`sort | uniq -c | sort -rn` 是一个经典组合：`uniq` 只合并**相邻**的重复行，所以要先 `sort`。

In [ ]:
%%bash
cd /tmp/linux-demo
grep ERROR train.log | cut -d' ' -f2 | sort | uniq -c | sort -rn

第一列就是出现次数。

### tee：边看边存

In [ ]:
%%bash
cd /tmp/linux-demo
grep ERROR train.log | tee errors.txt | wc -l
echo ---
cat errors.txt

## 动手试试

基于上面生成的 `/tmp/linux-demo/train.log`，不计分：

1. 把所有 `[INFO]` 行保存到 `info.txt`，再把所有 `[WARN]` 行追加到同一个文件
2. 运行 `ls /tmp/linux-demo /no/such/dir`，只把错误信息存到 `err.log`
3. 用一条管道输出出现次数最多的 ERROR 类型及其次数（只输出一行）

<details>
<summary>参考答案</summary>

```bash
cd /tmp/linux-demo
grep INFO train.log > info.txt
grep WARN train.log >> info.txt
ls /tmp/linux-demo /no/such/dir 2> err.log
grep ERROR train.log | cut -d' ' -f2 | sort | uniq -c | sort -rn | head -n 1
```

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 说出 stdin、stdout、stderr 的区别 | 三种标准流 |
| 区分 `>`、`>>`、`2>`、`2>&1` | 重定向 |
| 用管道组合命令，并逐段调试 | 管道 |
| 用 `sort`、`uniq`、`cut`、`tee` 处理文本 | 常用工具 |

下一节：[05-用户与权限](./05-用户与权限.ipynb)